# 🔍 Gold Debug Notebook — Stock Lakehouse ETL

**Mục đích:** Query từng bảng Bronze → Silver → Gold bằng DuckDB trực tiếp.  
Mỗi block độc lập, chạy từng ô để debug schema thực tế trước khi chạy SQLMesh.

**Workflow:**
```
BLOCK 0  → Setup: kết nối DuckDB + S3/MinIO
BLOCK 1  → Kiểm tra MinIO buckets / folder structure
BLOCK 2  → Bronze: query từng bảng, xem schema thực tế
BLOCK 3  → Silver: query từng bảng, xem schema thực tế
BLOCK 4  → Test SQL của từng Gold model (dry-run)
BLOCK 5  → Chạy SQLMesh programmatically từng bước
BLOCK 6  → Verify kết quả Gold trong DuckDB
```

---
## BLOCK 0 — Setup: DuckDB + S3/MinIO connection

In [ ]:
# ===========================================================================
# BLOCK 0A — Imports & project path
# ===========================================================================
import os
import sys
from pathlib import Path

import duckdb
import pandas as pd
import polars as pl
from dotenv import load_dotenv

# Chạy notebook từ bất kỳ thư mục nào cũng tìm được project root
PROJECT_ROOT = Path(globals().get('__vsc_ipynb_file__', __file__) if '__file__' in dir() else '.').resolve()
# Tìm project root theo marker file
for p in [PROJECT_ROOT] + list(PROJECT_ROOT.parents):
    if (p / 'pytest.ini').exists() or (p / 'scripts' / 'deploy_full_pipeline.py').exists():
        PROJECT_ROOT = p
        break

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / '.env')

print(f'✅ Project root: {PROJECT_ROOT}')
print(f'✅ Working dir:  {os.getcwd()}')

In [ ]:
# ===========================================================================
# BLOCK 0B — Đọc credentials từ .env
# ===========================================================================
_raw_endpoint = os.getenv('S3_ENDPOINT', 'http://localhost:9000')

# DuckDB httpfs cần HOST:PORT (không có http://)
S3_ENDPOINT    = _raw_endpoint.split('://')[-1]          # 'localhost:9000'
# s3fs/boto3 cần URL đầy đủ
S3_ENDPOINT_URL = _raw_endpoint if '://' in _raw_endpoint else f'http://{_raw_endpoint}'

S3_KEY    = os.getenv('AWS_ACCESS_KEY_ID',     os.getenv('MINIO_ROOT_USER',     'minioadmin'))
S3_SECRET = os.getenv('AWS_SECRET_ACCESS_KEY', os.getenv('MINIO_ROOT_PASSWORD', 'minioadmin_secure_123@#'))
BUCKET    = os.getenv('LAKEHOUSE_BUCKET', 'lakehouse')
BASE_PATH = f's3://{BUCKET}'

print(f'S3_ENDPOINT     : {S3_ENDPOINT}')
print(f'S3_ENDPOINT_URL : {S3_ENDPOINT_URL}')
print(f'S3_KEY          : {S3_KEY}')
print(f'BASE_PATH       : {BASE_PATH}')

In [ ]:
# ===========================================================================
# BLOCK 0C — Tạo DuckDB connection + cấu hình S3
# ===========================================================================
def make_duck(db_path: str = ':memory:') -> duckdb.DuckDBPyConnection:
    """Tạo DuckDB connection đã cấu hình sẵn httpfs/S3."""
    con = duckdb.connect(db_path)
    con.execute('INSTALL httpfs; LOAD httpfs;')
    con.execute(f"SET s3_endpoint='{S3_ENDPOINT}';")
    con.execute(f"SET s3_access_key_id='{S3_KEY}';")
    con.execute(f"SET s3_secret_access_key='{S3_SECRET}';")
    con.execute('SET s3_use_ssl=false;')
    con.execute("SET s3_url_style='path';")
    con.execute("SET s3_region='us-east-1';")
    return con

duck = make_duck()
print('✅ DuckDB in-memory connection ready')

# Quick connectivity test
try:
    duck.execute(f"SELECT COUNT(*) FROM read_parquet('{BASE_PATH}/bronze/**/*.parquet', hive_partitioning=true) LIMIT 0").fetchone()
    print('✅ MinIO reachable')
except Exception as e:
    print(f'❌ MinIO not reachable: {e}')
    print('   → Kiểm tra docker ps | grep minio, và credentials trong .env')

In [ ]:
# ===========================================================================
# BLOCK 0D — Helper functions dùng xuyên suốt notebook
# ===========================================================================
def q(sql: str, limit: int = 10) -> pd.DataFrame:
    """Chạy query, trả về DataFrame. Tự động thêm LIMIT."""
    try:
        sql_exec = sql.strip().rstrip(';')
        if limit and 'LIMIT' not in sql_exec.upper():
            sql_exec += f' LIMIT {limit}'
        return duck.execute(sql_exec).df()
    except Exception as e:
        print(f'❌ Query error: {e}')
        return pd.DataFrame({'error': [str(e)]})

def schema(path: str) -> pd.DataFrame:
    """Xem schema (column names + dtypes) của Parquet path."""
    return q(f"DESCRIBE SELECT * FROM read_parquet('{path}', hive_partitioning=true)", limit=0)

def count(path: str) -> int:
    """Đếm số rows trong Parquet path."""
    try:
        return duck.execute(f"SELECT COUNT(*) FROM read_parquet('{path}', hive_partitioning=true)").fetchone()[0]
    except Exception as e:
        return f'ERROR: {e}'

def peek(path: str, n: int = 5) -> pd.DataFrame:
    """Xem n rows đầu từ Parquet path."""
    return q(f"SELECT * FROM read_parquet('{path}', hive_partitioning=true)", limit=n)

def safe_sql(sql: str, label: str = '') -> pd.DataFrame:
    """Chạy SQL, print lỗi chi tiết thay vì crash."""
    try:
        result = duck.execute(sql.rstrip(';')).df()
        print(f'✅ {label}: {len(result)} rows')
        return result
    except duckdb.BinderException as e:
        print(f'❌ Binder Error [{label}]: {e}')
        print('   → Cột không tồn tại trong schema. Chạy schema() để kiểm tra.')
    except duckdb.IOException as e:
        print(f'❌ IO Error [{label}]: {e}')
        print('   → File không tồn tại hoặc MinIO không kết nối được.')
    except Exception as e:
        print(f'❌ Error [{label}]: {type(e).__name__}: {e}')
    return pd.DataFrame()

print('✅ Helper functions loaded: q(), schema(), count(), peek(), safe_sql()')

---
## BLOCK 1 — Kiểm tra MinIO: folder structure & partitions

In [ ]:
# ===========================================================================
# BLOCK 1A — Liệt kê tất cả Parquet files trong MinIO
# ===========================================================================
import s3fs

fs = s3fs.S3FileSystem(
    key=S3_KEY,
    secret=S3_SECRET,
    client_kwargs={'endpoint_url': S3_ENDPOINT_URL, 'region_name': 'us-east-1'},
    config_kwargs={'signature_version': 's3v4'},
)

def list_parquet_files(prefix: str = 'lakehouse', depth: int = 4) -> pd.DataFrame:
    """Liệt kê tất cả .parquet files trong MinIO bucket."""
    try:
        all_files = fs.glob(f'{prefix}/**/*.parquet')
        rows = []
        for f in all_files:
            parts = f.split('/')
            layer  = parts[1] if len(parts) > 1 else ''
            table  = parts[2] if len(parts) > 2 else ''
            try:
                size_kb = round(fs.info(f)['size'] / 1024, 1)
            except:
                size_kb = 0
            rows.append({'path': f, 'layer': layer, 'table': table, 'size_kb': size_kb})
        return pd.DataFrame(rows)
    except Exception as e:
        print(f'❌ {e}')
        return pd.DataFrame()

files_df = list_parquet_files()
print(f'Total Parquet files: {len(files_df)}')
files_df

In [ ]:
# ===========================================================================
# BLOCK 1B — Summary theo layer/table
# ===========================================================================
if not files_df.empty:
    summary = files_df.groupby(['layer', 'table']).agg(
        file_count=('path', 'count'),
        total_size_kb=('size_kb', 'sum')
    ).reset_index().sort_values(['layer', 'table'])
    display(summary)
else:
    print('Không có files — kiểm tra lại MinIO connection')

---
## BLOCK 2 — Bronze layer: schema + sample data

In [ ]:
# ===========================================================================
# BLOCK 2A — Bronze: stock_prices
# ===========================================================================
BRONZE_STOCK = f'{BASE_PATH}/bronze/stock_prices/**/*.parquet'
print(f'Path: {BRONZE_STOCK}')
print(f'Row count: {count(BRONZE_STOCK)}')
print('\n--- SCHEMA ---')
display(schema(BRONZE_STOCK))
print('\n--- SAMPLE ---')
display(peek(BRONZE_STOCK, 5))

In [ ]:
# ===========================================================================
# BLOCK 2B — Bronze: company_profile
# ===========================================================================
BRONZE_COMPANY = f'{BASE_PATH}/bronze/company_profile/**/*.parquet'
print(f'Row count: {count(BRONZE_COMPANY)}')
display(schema(BRONZE_COMPANY))
display(peek(BRONZE_COMPANY))

In [ ]:
# ===========================================================================
# BLOCK 2C — Bronze: financial_ratios
# ===========================================================================
BRONZE_RATIOS = f'{BASE_PATH}/bronze/financial_ratios/**/*.parquet'
print(f'Row count: {count(BRONZE_RATIOS)}')
display(schema(BRONZE_RATIOS))
display(peek(BRONZE_RATIOS))

In [ ]:
# ===========================================================================
# BLOCK 2D — Bronze: income_statement (quarter + year)
# ===========================================================================
for rtype in ['quarter', 'year']:
    path = f"{BASE_PATH}/bronze/income_statement_{rtype}/**/*.parquet"
    n = count(path)
    print(f'income_statement_{rtype}: {n} rows')
    if n and n != 'ERROR: 0':
        display(schema(path))
        display(peek(path, 3))
    print()

In [ ]:
# ===========================================================================
# BLOCK 2E — Bronze: balance_sheet (quarter + year)
# ===========================================================================
for rtype in ['quarter', 'year']:
    path = f"{BASE_PATH}/bronze/balance_sheet_{rtype}/**/*.parquet"
    n = count(path)
    print(f'balance_sheet_{rtype}: {n} rows')
    if n and n != 'ERROR: 0':
        display(schema(path))
        display(peek(path, 3))
    print()

In [ ]:
# ===========================================================================
# BLOCK 2F — Bronze: industry_sectors + market_type_sectors
# ===========================================================================
for tbl in ['industry_sectors', 'market_type_sectors']:
    path = f"{BASE_PATH}/bronze/{tbl}/**/*.parquet"
    n = count(path)
    print(f'{tbl}: {n} rows')
    if n and str(n) != 'ERROR: 0':
        display(schema(path))
        display(peek(path, 3))
    print()

In [ ]:
# ===========================================================================
# BLOCK 2G — Bronze: tất cả bảng — quick count summary
# ===========================================================================
BRONZE_TABLES = [
    'stock_prices',
    'company_profile',
    'financial_ratios',
    'financial_report_summary',
    'business_plan',
    'income_statement_quarter',
    'income_statement_year',
    'balance_sheet_quarter',
    'balance_sheet_year',
    'industry_sectors',
    'market_type_sectors',
    'industry_info_summary_info',
    'industry_info_financial_info',
    'industry_info_fund_info',
]

bronze_summary = []
for tbl in BRONZE_TABLES:
    path = f'{BASE_PATH}/bronze/{tbl}/**/*.parquet'
    n = count(path)
    bronze_summary.append({'table': tbl, 'rows': n})

pd.DataFrame(bronze_summary)

---
## BLOCK 3 — Silver layer: schema + sample data

In [ ]:
# ===========================================================================
# BLOCK 3A — Silver: fact_stock_price
# ===========================================================================
SIL_STOCK = f'{BASE_PATH}/silver/fact_stock_price/**/*.parquet'
print(f'fact_stock_price: {count(SIL_STOCK)} rows')
print('\n--- SCHEMA ---')
display(schema(SIL_STOCK))
print('\n--- SAMPLE ---')
display(peek(SIL_STOCK, 5))

In [ ]:
# ===========================================================================
# BLOCK 3B — Silver: dim_company
# ===========================================================================
SIL_COMPANY = f'{BASE_PATH}/silver/dim_company/**/*.parquet'
print(f'dim_company: {count(SIL_COMPANY)} rows')
display(schema(SIL_COMPANY))
display(peek(SIL_COMPANY, 5))

# Kiểm tra giá trị is_current thực tế
print('\n--- is_current values ---')
display(q(f"SELECT is_current, COUNT(*) AS cnt FROM read_parquet('{SIL_COMPANY}', hive_partitioning=true) GROUP BY 1", limit=0))

In [ ]:
# ===========================================================================
# BLOCK 3C — Silver: dim_industry
# ===========================================================================
SIL_INDUSTRY = f'{BASE_PATH}/silver/dim_industry/**/*.parquet'
print(f'dim_industry: {count(SIL_INDUSTRY)} rows')
display(schema(SIL_INDUSTRY))
display(peek(SIL_INDUSTRY, 5))

In [ ]:
# ===========================================================================
# BLOCK 3D — Silver: dim_market_type
# ===========================================================================
SIL_MARKET = f'{BASE_PATH}/silver/dim_market_type/**/*.parquet'
print(f'dim_market_type: {count(SIL_MARKET)} rows')
display(schema(SIL_MARKET))
display(peek(SIL_MARKET, 5))

# Kiểm tra market_type_code có tồn tại không
print('\n--- Distinct market_type_code values ---')
display(q(f"SELECT DISTINCT market_type_code, market_type_name FROM read_parquet('{SIL_MARKET}', hive_partitioning=true)", limit=0))

In [ ]:
# ===========================================================================
# BLOCK 3E — Silver: tất cả bảng — quick count summary
# ===========================================================================
SILVER_TABLES = [
    'fact_stock_price',
    'dim_company',
    'dim_industry',
    'dim_market_type',
    'fact_financial_metrics',
    'fact_financial_report',
    'fact_business_plan',
    'fact_income_statement',
    'fact_balance_sheet',
    'fact_industry_summary',
]

silver_summary = []
for tbl in SILVER_TABLES:
    path = f'{BASE_PATH}/silver/{tbl}/**/*.parquet'
    n = count(path)
    # Lấy danh sách columns
    try:
        cols = duck.execute(
            f"SELECT column_name FROM (DESCRIBE SELECT * FROM read_parquet('{path}', hive_partitioning=true)) LIMIT 0"
        ).df()['column_name'].tolist()
        col_str = ', '.join(cols[:8]) + ('...' if len(cols) > 8 else '')
    except:
        col_str = 'N/A'
    silver_summary.append({'table': tbl, 'rows': n, 'columns': col_str})

pd.DataFrame(silver_summary)

---
## BLOCK 4 — Test từng Gold SQL model (dry-run)

Mỗi ô chạy SQL thủ công của từng model — không qua SQLMesh.  
Nếu lỗi Binder Error → column không tồn tại → sửa SQL trước khi chạy SQLMesh.

In [ ]:
# ===========================================================================
# BLOCK 4A — Test: gold_dim_company_current
# Tương đương: platforms/processing/sqlmesh/models/gold_dim_company_current.sql
# ===========================================================================
SQL_DIM_COMPANY = f"""
SELECT
  company_key,
  symbol,
  full_name                            AS company_name,
  full_name,
  english_name,
  short_name,
  address,
  website,
  email_address,
  established_date,
  listed_date,
  listed_volume,
  market_capitalization,
  effective_date,
  CURRENT_TIMESTAMP AS _updated_at
FROM read_parquet('{BASE_PATH}/silver/dim_company/**/*.parquet', hive_partitioning=true)
WHERE COALESCE(TRY_CAST(is_current AS BOOLEAN), true) = true
LIMIT 5
"""

result_dim_company = safe_sql(SQL_DIM_COMPANY, label='gold_dim_company_current')
display(result_dim_company)

In [ ]:
# ===========================================================================
# BLOCK 4B — Test: gold_fct_bs_assets
# Tương đương: platforms/processing/sqlmesh/models/gold_fct_bs_assets.sql
# ===========================================================================
SQL_BS_ASSETS = f"""
SELECT
  symbol,
  time_report_type,
  report_type                          AS financial_report_type,
  COALESCE(TRY_CAST(year AS VARCHAR), LEFT(ingest_date, 4)) AS year,
  NULL                                 AS period,
  TRY_CAST(NULL AS DECIMAL(30,4))      AS cash_and_valuables,
  TRY_CAST(NULL AS DECIMAL(30,4))      AS total_assets,
  CURRENT_TIMESTAMP                    AS update_time,
  CURRENT_TIMESTAMP                    AS _updated_at
FROM read_parquet('{BASE_PATH}/silver/fact_balance_sheet/**/*.parquet', hive_partitioning=true)
LIMIT 5
"""

result_bs = safe_sql(SQL_BS_ASSETS, label='gold_fct_bs_assets')
display(result_bs)

In [ ]:
# ===========================================================================
# BLOCK 4C — Test: gold_fct_is_core_income
# Tương đương: platforms/processing/sqlmesh/models/gold_fct_is_core_income.sql
# ===========================================================================
SQL_IS_INCOME = f"""
SELECT
  symbol,
  time_report_type,
  report_type                          AS financial_report_type,
  COALESCE(TRY_CAST(year AS VARCHAR), LEFT(ingest_date, 4)) AS year,
  NULL                                 AS period,
  TRY_CAST(NULL AS DECIMAL(30,4))      AS net_interest_income,
  TRY_CAST(NULL AS DECIMAL(30,4))      AS profit_before_tax,
  TRY_CAST(NULL AS DECIMAL(30,4))      AS profit_after_tax,
  CURRENT_TIMESTAMP                    AS update_time,
  CURRENT_TIMESTAMP                    AS _updated_at
FROM read_parquet('{BASE_PATH}/silver/fact_income_statement/**/*.parquet', hive_partitioning=true)
LIMIT 5
"""

result_is = safe_sql(SQL_IS_INCOME, label='gold_fct_is_core_income')
display(result_is)

In [ ]:
# ===========================================================================
# BLOCK 4D — Test: gold_fct_trading_ohlcv — từng phần
# Tương đương: platforms/processing/sqlmesh/models/gold_fct_trading_ohlcv.sql
# ===========================================================================

# --- Step 1: fact_stock_price base ---
print('Step 1: fact_stock_price base')
SQL_BASE = f"""
SELECT trade_key, symbol, date, close_price, open_price,
       high_price, low_price, volume, foreign_buy, foreign_sell, foreign_value,
       year, month
FROM read_parquet('{BASE_PATH}/silver/fact_stock_price/**/*.parquet', hive_partitioning=true)
LIMIT 5
"""
result_base = safe_sql(SQL_BASE, label='base')
display(result_base)

In [ ]:
# --- Step 2: dim_company subquery ---
print('Step 2: dim_company subquery')
SQL_DIM_C = f"""
SELECT symbol, full_name
FROM read_parquet('{BASE_PATH}/silver/dim_company/**/*.parquet', hive_partitioning=true)
WHERE COALESCE(TRY_CAST(is_current AS BOOLEAN), true) = true
LIMIT 5
"""
result_dim_c = safe_sql(SQL_DIM_C, label='dim_company')
display(result_dim_c)

In [ ]:
# --- Step 3: dim_industry subquery ---
print('Step 3: dim_industry subquery')
SQL_DIM_I = f"""
SELECT DISTINCT symbol, industry_name
FROM read_parquet('{BASE_PATH}/silver/dim_industry/**/*.parquet', hive_partitioning=true)
WHERE COALESCE(TRY_CAST(is_current AS BOOLEAN), true) = true
LIMIT 5
"""
result_dim_i = safe_sql(SQL_DIM_I, label='dim_industry')
display(result_dim_i)

In [ ]:
# --- Step 4: dim_market_type subquery ---
print('Step 4: dim_market_type subquery')
SQL_DIM_M = f"""
SELECT DISTINCT symbol, market_type_code
FROM read_parquet('{BASE_PATH}/silver/dim_market_type/**/*.parquet', hive_partitioning=true)
LIMIT 5
"""
result_dim_m = safe_sql(SQL_DIM_M, label='dim_market_type')
display(result_dim_m)

In [ ]:
# --- Step 5: FULL JOIN query ---
print('Step 5: FULL JOIN (gold_fct_trading_ohlcv)')
SQL_OHLCV = f"""
SELECT
  t.trade_key,
  t.symbol,
  c.full_name                          AS company_name,
  i.industry_name,
  m.market_type_code                   AS market_type,
  TRY_CAST(t.date AS DATE)             AS trade_date,
  TRY_CAST(t.close_price AS DOUBLE)    AS close_price,
  TRY_CAST(t.open_price  AS DOUBLE)    AS open_price,
  TRY_CAST(t.high_price  AS DOUBLE)    AS high_price,
  TRY_CAST(t.low_price   AS DOUBLE)    AS low_price,
  TRY_CAST(t.volume      AS BIGINT)    AS volume,
  TRY_CAST(t.foreign_buy AS BIGINT)    AS foreign_buy,
  TRY_CAST(t.foreign_sell AS BIGINT)   AS foreign_sell,
  TRY_CAST(t.foreign_value AS DOUBLE)  AS foreign_net_value,
  t.year,
  t.month,
  CURRENT_TIMESTAMP                    AS _updated_at
FROM read_parquet('{BASE_PATH}/silver/fact_stock_price/**/*.parquet', hive_partitioning=true) AS t
LEFT JOIN (
  SELECT symbol, full_name
  FROM read_parquet('{BASE_PATH}/silver/dim_company/**/*.parquet', hive_partitioning=true)
  WHERE COALESCE(TRY_CAST(is_current AS BOOLEAN), true) = true
) AS c ON t.symbol = c.symbol
LEFT JOIN (
  SELECT DISTINCT symbol, industry_name
  FROM read_parquet('{BASE_PATH}/silver/dim_industry/**/*.parquet', hive_partitioning=true)
  WHERE COALESCE(TRY_CAST(is_current AS BOOLEAN), true) = true
) AS i ON t.symbol = i.symbol
LEFT JOIN (
  SELECT DISTINCT symbol, market_type_code
  FROM read_parquet('{BASE_PATH}/silver/dim_market_type/**/*.parquet', hive_partitioning=true)
) AS m ON t.symbol = m.symbol
LIMIT 10
"""

result_ohlcv = safe_sql(SQL_OHLCV, label='gold_fct_trading_ohlcv')
display(result_ohlcv)

In [ ]:
# ===========================================================================
# BLOCK 4E — Data quality check: NULL ratio cho từng join
# ===========================================================================
if not result_ohlcv.empty:
    null_pct = (result_ohlcv.isnull().sum() / len(result_ohlcv) * 100).round(1)
    print('NULL % per column:')
    display(null_pct[null_pct > 0].to_frame('null_%'))
    
    # Kiểm tra join hit rate
    total = len(result_ohlcv)
    matched_company  = result_ohlcv['company_name'].notna().sum()
    matched_industry = result_ohlcv['industry_name'].notna().sum()
    matched_market   = result_ohlcv['market_type'].notna().sum()
    print(f'\nJoin hit rates (out of {total} rows):')
    print(f'  company_name  : {matched_company}/{total} ({matched_company/total*100:.0f}%)')
    print(f'  industry_name : {matched_industry}/{total} ({matched_industry/total*100:.0f}%)')
    print(f'  market_type   : {matched_market}/{total} ({matched_market/total*100:.0f}%)')

---
## BLOCK 5 — Chạy SQLMesh programmatically từng bước

> **Trước khi chạy:** Đảm bảo BLOCK 4 đã pass hết (không có BinderError).

In [ ]:
# ===========================================================================
# BLOCK 5A — Import SqlMeshEngine và kiểm tra config
# ===========================================================================
import os

# Set env vars cho SQLMesh TRƯỚC khi import (vì deploy_full_pipeline.py cũng làm vậy)
os.environ['S3_ENDPOINT']          = S3_ENDPOINT       # HOST:PORT, không có http://
os.environ['AWS_ACCESS_KEY_ID']    = S3_KEY
os.environ['AWS_SECRET_ACCESS_KEY']= S3_SECRET

from platforms.processing.sqlmesh.sqlmesh_engine import SqlMeshConfig, SqlMeshEngine

SQLMESH_PATH = str(PROJECT_ROOT / 'platforms' / 'processing' / 'sqlmesh')
print(f'SQLMesh path: {SQLMESH_PATH}')
print(f'config.yaml:  {SQLMESH_PATH}/config.yaml')

# Kiểm tra config.yaml tồn tại
import yaml
with open(f'{SQLMESH_PATH}/config.yaml') as f:
    cfg = yaml.safe_load(f)
print('\nconfig.yaml content:')
print(yaml.dump(cfg, default_flow_style=False))

In [ ]:
# ===========================================================================
# BLOCK 5B — Khởi tạo SQLMesh Context
# ===========================================================================
import logging
logging.basicConfig(level=logging.WARNING)  # Giảm noise, chỉ hiện WARNING+

try:
    sqlmesh_cfg = SqlMeshConfig(project_path=SQLMESH_PATH, gateway='local_duckdb')
    engine = SqlMeshEngine(config=sqlmesh_cfg)
    print('✅ SQLMesh Context khởi tạo thành công')
    print(f'   Models: {list(engine.context.models.keys())}')
except Exception as e:
    print(f'❌ SQLMesh Context failed: {type(e).__name__}: {e}')
    raise

In [ ]:
# ===========================================================================
# BLOCK 5C — Kiểm tra S3 config đã được inject vào SQLMesh adapter
# ===========================================================================
try:
    adapter = engine.context._engine_adapter
    # Thử đọc một row từ silver để verify httpfs hoạt động
    test_sql = f"SELECT COUNT(*) FROM read_parquet('{BASE_PATH}/silver/fact_stock_price/**/*.parquet', hive_partitioning=true)"
    n = adapter.execute(test_sql).fetchone()[0]
    print(f'✅ SQLMesh adapter có thể đọc MinIO. fact_stock_price: {n} rows')
except Exception as e:
    print(f'❌ SQLMesh adapter không đọc được MinIO: {e}')
    print('   → Kiểm tra _configure_s3() đã inject đúng chưa')

In [ ]:
# ===========================================================================
# BLOCK 5D — Render model SQL (xem DuckDB sẽ execute gì)
# ===========================================================================
from sqlmesh import Context

ctx = engine.context

for model_name in ctx.models:
    try:
        model = ctx.models[model_name]
        print(f'\n{'='*60}')
        print(f'MODEL: {model_name}')
        print(f'Kind:  {model.kind}')
        print(f'Cron:  {getattr(model, "cron", "N/A")}')
        # Render SQL — xem query thực sự
        rendered = model.render_query()
        print(f'\nRendered SQL (first 500 chars):')
        print(str(rendered)[:500])
    except Exception as e:
        print(f'  ❌ render error: {e}')

In [ ]:
# ===========================================================================
# BLOCK 5E — SQLMesh Plan (xem diff, KHÔNG apply)
# ===========================================================================
ENV = 'dev'   # Đổi sang 'prod' khi cần

try:
    # no_prompts=True, auto_apply=False → chỉ xem plan, không chạy
    plan = ctx.plan(
        environment=ENV,
        auto_apply=False,
        no_prompts=True,
    )
    print(f'✅ Plan created for environment: {ENV}')
    print(f'   New snapshots    : {len(plan.new_snapshots)}')
    print(f'   Modified         : {len(plan.modified_snapshots)}')
    print(f'   Directly modified: {len(plan.directly_modified)}')
except Exception as e:
    print(f'❌ Plan failed: {type(e).__name__}: {e}')

In [ ]:
# ===========================================================================
# BLOCK 5F — SQLMesh Plan + APPLY (chạy thực sự)
# Chỉ chạy ô này sau khi BLOCK 4 đã pass hết!
# ===========================================================================
print('⚠️  Ô này sẽ THỰC SỰ CHẠY SQLMesh plan + backfill.')
print('   Chỉ chạy sau khi tất cả BLOCK 4 đã pass.')

CONFIRM = True   # Đổi thành True để chạy

if CONFIRM:
    try:
        result = engine.plan(environment=ENV)
        print('✅ Plan applied successfully')
        print(f'   Result: {result}')
    except Exception as e:
        print(f'❌ Plan failed: {type(e).__name__}: {e}')
        import traceback
        traceback.print_exc()
else:
    print('Bỏ qua — CONFIRM = False')

In [ ]:
# ===========================================================================
# BLOCK 5G — SQLMesh Run (incremental, sau khi plan đã apply)
# ===========================================================================
RUN_DATE = '2026-07-20'   # Đổi theo ngày cần chạy
CONFIRM_RUN = True

if CONFIRM_RUN:
    try:
        engine.run(environment=ENV, start=RUN_DATE, end=RUN_DATE)
        print(f'✅ SQLMesh run completed for {RUN_DATE}')
    except Exception as e:
        print(f'❌ Run failed: {type(e).__name__}: {e}')
        import traceback
        traceback.print_exc()

---
## BLOCK 6 — Verify Gold layer trong DuckDB

In [ ]:
# ===========================================================================
# BLOCK 6A — Kết nối DuckDB persistent file (nơi SQLMesh ghi kết quả)
# ===========================================================================
LAKEHOUSE_DB = str(PROJECT_ROOT / 'stock_lakehouse.db')

duck_gold = make_duck(LAKEHOUSE_DB)
print(f'✅ Connected to: {LAKEHOUSE_DB}')

# List schemas
schemas_df = duck_gold.execute("SHOW ALL TABLES").df()
display(schemas_df)

In [ ]:
# ===========================================================================
# BLOCK 6B — Query từng Gold table
# ===========================================================================
GOLD_TABLES = [
    ('stock_lakehouse', 'gold', 'dim_company_current'),
    ('stock_lakehouse', 'gold', 'fct_trading_ohlcv'),
    ('stock_lakehouse', 'gold', 'fct_bs_assets'),
    ('stock_lakehouse', 'gold', 'fct_is_core_income'),
]

gold_summary = []
for catalog, schema_name, table in GOLD_TABLES:
    try:
        n = duck_gold.execute(f'SELECT COUNT(*) FROM "{catalog}"."{schema_name}"."{table}"').fetchone()[0]
        gold_summary.append({'table': f'{schema_name}.{table}', 'rows': n, 'status': '✅'})
    except Exception as e:
        gold_summary.append({'table': f'{schema_name}.{table}', 'rows': 0, 'status': f'❌ {str(e)[:60]}'})

pd.DataFrame(gold_summary)

In [ ]:
# ===========================================================================
# BLOCK 6C — Sample data từ gold.fct_trading_ohlcv
# ===========================================================================
try:
    df_gold = duck_gold.execute(
        'SELECT * FROM stock_lakehouse.gold.fct_trading_ohlcv LIMIT 20'
    ).df()
    print(f'fct_trading_ohlcv: {len(df_gold)} sample rows')
    display(df_gold)
except Exception as e:
    print(f'❌ {e}')

In [ ]:
# ===========================================================================
# BLOCK 6D — Sample data từ gold.dim_company_current
# ===========================================================================
try:
    df_gold_co = duck_gold.execute(
        'SELECT * FROM stock_lakehouse.gold.dim_company_current LIMIT 10'
    ).df()
    display(df_gold_co)
except Exception as e:
    print(f'❌ {e}')

In [ ]:
# ===========================================================================
# BLOCK 6E — Phân tích dữ liệu Gold: OHLCV theo symbol
# ===========================================================================
try:
    agg = duck_gold.execute("""
        SELECT
          symbol,
          company_name,
          industry_name,
          market_type,
          COUNT(*)                    AS trading_days,
          MIN(trade_date)             AS first_date,
          MAX(trade_date)             AS last_date,
          ROUND(AVG(close_price), 2)  AS avg_close,
          MAX(close_price)            AS max_close,
          MIN(close_price)            AS min_close
        FROM stock_lakehouse.gold.fct_trading_ohlcv
        GROUP BY symbol, company_name, industry_name, market_type
        ORDER BY trading_days DESC
    """).df()
    display(agg)
except Exception as e:
    print(f'❌ {e}')

---
## BLOCK 7 — Adhoc debug: kiểm tra cột cụ thể

In [ ]:
# ===========================================================================
# BLOCK 7A — Kiểm tra một cột có tồn tại không trước khi dùng trong SQL
# Dùng khi gặp BinderError: Referenced column X not found
# ===========================================================================
def check_column(parquet_path: str, column_name: str) -> None:
    """Kiểm tra cột có tồn tại trong Parquet file không."""
    try:
        schema_df = duck.execute(
            f"DESCRIBE SELECT * FROM read_parquet('{parquet_path}', hive_partitioning=true)"
        ).df()
        cols = schema_df['column_name'].tolist()
        if column_name in cols:
            dtype = schema_df[schema_df['column_name'] == column_name]['column_type'].iloc[0]
            print(f'✅ Column "{column_name}" EXISTS: dtype={dtype}')
            # Sample giá trị
            sample = duck.execute(
                f"SELECT DISTINCT {column_name} FROM read_parquet('{parquet_path}', hive_partitioning=true) LIMIT 5"
            ).df()
            print('   Sample values:', sample[column_name].tolist())
        else:
            print(f'❌ Column "{column_name}" NOT FOUND')
            print(f'   Available columns: {cols}')
    except Exception as e:
        print(f'❌ Error: {e}')

# Ví dụ sử dụng:
check_column(f'{BASE_PATH}/silver/dim_industry/**/*.parquet', 'industry_name')
check_column(f'{BASE_PATH}/silver/dim_market_type/**/*.parquet', 'market_type_code')
check_column(f'{BASE_PATH}/silver/dim_company/**/*.parquet', 'is_current')

In [ ]:
# ===========================================================================
# BLOCK 7B — Kiểm tra dtype của tất cả cột trong một bảng
# Dùng khi gặp lỗi dtype mismatch (datetime[ms] vs String)
# ===========================================================================
def dtype_report(parquet_path: str, table_label: str = '') -> pd.DataFrame:
    """Báo cáo dtype thực tế của tất cả cột."""
    try:
        df = duck.execute(
            f"DESCRIBE SELECT * FROM read_parquet('{parquet_path}', hive_partitioning=true)"
        ).df()
        # Flag cột datetime (potential type mismatch issue)
        df['is_datetime'] = df['column_type'].str.contains('TIMESTAMP|DATE|TIME', case=False, na=False)
        df['is_numeric']  = df['column_type'].str.contains('INT|FLOAT|DOUBLE|DECIMAL|BIGINT', case=False, na=False)
        if table_label:
            print(f'=== {table_label} ===')
        n_dt = df['is_datetime'].sum()
        if n_dt > 0:
            print(f'⚠️  {n_dt} datetime columns found — Silver transform cần cast sang Utf8!')
        return df[['column_name', 'column_type', 'is_datetime', 'is_numeric']]
    except Exception as e:
        print(f'❌ {e}')
        return pd.DataFrame()

# Kiểm tra các bảng Bronze — đây là nguồn gốc của datetime dtype issues
for tbl in ['stock_prices', 'income_statement_quarter', 'balance_sheet_quarter']:
    path = f'{BASE_PATH}/bronze/{tbl}/**/*.parquet'
    df_report = dtype_report(path, f'bronze/{tbl}')
    if not df_report.empty:
        display(df_report)
    print()

In [ ]:
# ===========================================================================
# BLOCK 7C — Test _normalise_dtypes logic trực tiếp
# ===========================================================================
import polars as pl

def normalise_dtypes_test(parquet_path: str) -> pl.DataFrame:
    """Simulate _normalise_dtypes() từ SilverProcessor._read_bronze()."""
    # Đọc qua DuckDB
    raw = duck.execute(
        f"SELECT * FROM read_parquet('{parquet_path}', hive_partitioning=true) LIMIT 100"
    ).pl()
    
    print(f'Before normalise: {raw.dtypes}')
    
    KEEP_NUMERIC = (
        pl.Int8, pl.Int16, pl.Int32, pl.Int64,
        pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64,
        pl.Float32, pl.Float64,
    )
    cast_exprs = []
    for col_name, dtype in zip(raw.columns, raw.dtypes):
        if isinstance(dtype, KEEP_NUMERIC):
            continue
        if dtype in (pl.Utf8, pl.String):
            continue
        print(f'  Cast: {col_name} ({dtype}) → Utf8')
        cast_exprs.append(pl.col(col_name).cast(pl.Utf8, strict=False).alias(col_name))
    
    if cast_exprs:
        raw = raw.with_columns(cast_exprs)
    
    print(f'After normalise:  {raw.dtypes}')
    return raw

# Test với stock_prices (bảng có nguy cơ datetime cao nhất)
try:
    df_norm = normalise_dtypes_test(f'{BASE_PATH}/bronze/stock_prices/**/*.parquet')
    display(df_norm.head(3))
except Exception as e:
    print(f'❌ {e}')

In [ ]:
# ===========================================================================
# BLOCK 7D — Chạy adhoc SQL (free-form debug)
# ===========================================================================
# Thay SQL_ADHOC bằng bất kỳ câu query nào cần debug
SQL_ADHOC = f"""
SELECT
  symbol,
  COUNT(*) AS cnt,
  MIN(date) AS first_date,
  MAX(date) AS last_date
FROM read_parquet('{BASE_PATH}/bronze/stock_prices/**/*.parquet', hive_partitioning=true)
GROUP BY symbol
ORDER BY cnt DESC
LIMIT 10
"""

result_adhoc = safe_sql(SQL_ADHOC, label='adhoc')
display(result_adhoc)